#### Combine the pollutants and states

In [2]:
import pandas as pd
import os
from pathlib import Path

# German states
states = [
    "Baden-Württemberg", "Bayern", "Berlin", "Brandenburg", "Bremen", "Hamburg",
    "Hessen", "Mecklenburg-Vorpommern", "Niedersachsen", "Nordrhein-Westfalen",
    "Rheinland-Pfalz", "Saarland", "Sachsen", "Sachsen-Anhalt",
    "Schleswig-Holstein", "Thüringen"
]

# Map for the file names
filename_to_state = {
    "baden-württemberg": "Baden-Württemberg",
    "bayern": "Bayern",
    "berlin": "Berlin",
    "brandenburg": "Brandenburg",
    "bremen": "Bremen",
    "hamburg": "Hamburg",
    "hessen": "Hessen",
    "mecklenburg-vorpommern": "Mecklenburg-Vorpommern",
    "niedersachsen": "Niedersachsen",
    "nordrhein-westfalen": "Nordrhein-Westfalen",
    "rheinland-pfalz": "Rheinland-Pfalz",
    "saarland": "Saarland",
    "sachsen": "Sachsen",
    "sachsen-anhalt": "Sachsen-Anhalt",
    "schleswig-holstein": "Schleswig-Holstein",
    "thüringen": "Thüringen"
}

# The relevant pollutants
pollutants = ["PM10", "PM25", "NO2"]

files = {}

for pollutant in pollutants:
    folder = Path(pollutant)
    # Find all CSV files in the folder
    csv_files = list(folder.glob("daily_avg_*_*.csv"))
    files[pollutant] = [str(f) for f in csv_files]

# Step 1: Load each pollutant and process
pollutant_dfs = {}

for pollutant, paths in files.items():
    dfs = []
    for path in paths:
        df = pd.read_csv(path)
        
        # Extract state key from filename
        state_key = os.path.basename(path).split("_")[2].lower()
        df["state"] = filename_to_state[state_key]
        
        # Rename columns
        df = df.rename(columns={
            "date start": "date",
            "value": pollutant.lower()  # lowercase: pm10, pm25, no2
        })
        
        dfs.append(df)
    
    # Concatenate all states for this pollutant
    pollutant_dfs[pollutant] = pd.concat(dfs, ignore_index=True)

# Step 2: Merge all pollutants on date + state
df_all = pollutant_dfs["PM10"]
for p in ["PM25", "NO2"]:
    df_all = df_all.merge(pollutant_dfs[p], on=["date", "state"], how = "outer")

# Step 3: Convert date column to datetime
df_all["date"] = pd.to_datetime(df_all["date"])

# Drop the time
df_all["date"] = pd.to_datetime(df_all["date"]).dt.date
# Reorder
df_all = df_all[["date", "state", "pm10", "pm25", "no2"]]
# Make sure it's date time
df_all["date"] = pd.to_datetime(df_all["date"])
df_all

,date,state,pm10,pm25,no2
0,2016-01-01,Baden-Württemberg,95.589744,NaN,26.515625
1,2016-01-01,Bayern,95.589744,55.555556,22.562202
2,2016-01-01,Berlin,120.777778,NaN,27.000000
3,2016-01-01,Brandenburg,48.950000,39.250000,17.065476
4,2016-01-01,Bremen,78.428571,47.333333,26.229167
...,...,...,...,...,...
59403,2026-03-01,Saarland,NaN,NaN,13.166667
59404,2026-03-01,Sachsen,NaN,NaN,8.217391
59405,2026-03-01,Sachsen-Anhalt,NaN,NaN,5.842995
59406,2026-03-01,Schleswig-Holstein,NaN,NaN,5.663462


#### Save new files

In [ ]:
## Monthly stats

monthly_state = (
    df_all
    .groupby(["state", pd.Grouper(key="date", freq="ME")])
    .mean()
    .reset_index()
)
# Rename 'date' → 'month' in monthly averages
monthly_state["date"] = monthly_state["date"].dt.to_period("M")
monthly_state = monthly_state.rename(columns={"date": "month"})

monthly_state


### Yearly stats
yearly_state = (
    df_all
    .groupby(["state", pd.Grouper(key="date", freq="YE")])
    .mean()
    .reset_index()
)

# Convert 'date' to year only
yearly_state["date"] = yearly_state["date"].dt.to_period("Y")

# Rename the column to 'year'
yearly_state = yearly_state.rename(columns={"date": "year"})
yearly_state

# Reorder daily columns
daily_state = df_all[["state", "date", "pm10", "pm25", "no2"]]
# Make sure 'date' column is datetime
daily_state['date'] = pd.to_datetime(daily_state['date'])
# Sort by state first, then date
daily_state = daily_state.sort_values(by=['state', 'date']).reset_index(drop=True)

# Saving files
daily_state.to_csv("States_Daily_Data.csv", index = False)
monthly_state.to_csv("States_Monthly_Data.csv", index = False)
yearly_state.to_csv("States_Yearly_Data.csv", index = False)